# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/magnito/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/magnito/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/magnito/projects/aie-projects-mk/AIE_MK/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/magnito/projects/aie-projects-mk/AIE_MK/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/magnito/projects/aie-projects-mk/AIE_MK/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'ca80e4'. Skipping!
Property 'summary' already exists in node '56e063'. Skipping!
Property 'summary' already exists in node '585b8d'. Skipping!
Property 'summary' already exists in node '47a859'. Skipping!
Property 'summary' already exists in node '86651e'. Skipping!
Property 'summary' already exists in node '101b2f'. Skipping!
Property 'summary' already exists in node 'fb0635'. Skipping!
Property 'summary' already exists in node '8b23fc'. Skipping!
Property 'summary' already exists in node '84aaec'. Skipping!
Property 'summary' already exists in node '6eaf5c'. Skipping!
Property 'summary' already exists in node 'cae2a2'. Skipping!
Property 'summary' already exists in node '2b0d1c'. Skipping!
Property 'summary' already exists in node '75d048'. Skipping!
Property 'summary' already exists in node 'c4ffce'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'ca80e4'. Skipping!
Property 'summary_embedding' already exists in node '56e063'. Skipping!
Property 'summary_embedding' already exists in node '101b2f'. Skipping!
Property 'summary_embedding' already exists in node '585b8d'. Skipping!
Property 'summary_embedding' already exists in node '8b23fc'. Skipping!
Property 'summary_embedding' already exists in node 'fb0635'. Skipping!
Property 'summary_embedding' already exists in node '47a859'. Skipping!
Property 'summary_embedding' already exists in node '2b0d1c'. Skipping!
Property 'summary_embedding' already exists in node '6eaf5c'. Skipping!
Property 'summary_embedding' already exists in node '84aaec'. Skipping!
Property 'summary_embedding' already exists in node 'c4ffce'. Skipping!
Property 'summary_embedding' already exists in node '86651e'. Skipping!
Property 'summary_embedding' already exists in node 'cae2a2'. Skipping!
Property 'summary_embedding' already exists in node '75d048'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [11]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

`SingleHopSpecificQuerySynthesizer` - creates simple questions that can be answered by taking some piece of information in the document.

`MultiHopAbstractQuerySynthesizer` - creates complex questions using reasoning across multiple pieces of information (focuses on abstract concepts).

`MultiHopSpecificQuerySynthesizer` - creates detailed questions, combines logical connection and factual recall to get specific facts from multiple sections.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How can I access detailed information about ac...,"[Chapter 1 Academic Years, Academic Calendars,...",The context indicates that the Knowledge Cente...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(a) about?,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,Chapter 3 what is it and how does it relate to...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,Can you explain what the Title IV refers to in...,[Non-Term Characteristics A program that measu...,The context indicates that Title IV programs a...,single_hop_specifc_query_synthesizer
4,What is Volume 7 about?,[both the credit or clock hours and the weeks ...,Volume 7 discusses the credit or clock hours a...,single_hop_specifc_query_synthesizer
5,Considering the requirements for clinical or p...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work that overlaps s...,multi_hop_abstract_query_synthesizer
6,How does participation in clinical or practicu...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Participation in clinical or practicum experie...,multi_hop_abstract_query_synthesizer
7,H0w do the reguLations and approval proccesses...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulations and approval processes for aca...,multi_hop_abstract_query_synthesizer
8,How do Volume 2 and Volume 7 relate to academi...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 discusses the academic year requireme...,multi_hop_specific_query_synthesizer
9,Can you explain how Volume 2 and Volume 8 rela...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...","Based on Volume 2, the academic year must incl...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '75d642'. Skipping!
Property 'summary' already exists in node '8f2214'. Skipping!
Property 'summary' already exists in node '38be61'. Skipping!
Property 'summary' already exists in node 'd8cc7b'. Skipping!
Property 'summary' already exists in node '26e922'. Skipping!
Property 'summary' already exists in node 'af4ef6'. Skipping!
Property 'summary' already exists in node '06bd36'. Skipping!
Property 'summary' already exists in node 'fd9a6f'. Skipping!
Property 'summary' already exists in node 'bc887b'. Skipping!
Property 'summary' already exists in node '1490f9'. Skipping!
Property 'summary' already exists in node '0317bf'. Skipping!
Property 'summary' already exists in node '03af45'. Skipping!
Property 'summary' already exists in node '4f9272'. Skipping!
Property 'summary' already exists in node '317765'. Skipping!


...

Property 'summary_embedding' already exists in node '75d642'. Skipping!
Property 'summary_embedding' already exists in node '8f2214'. Skipping!
Property 'summary_embedding' already exists in node '38be61'. Skipping!
Property 'summary_embedding' already exists in node 'af4ef6'. Skipping!
Property 'summary_embedding' already exists in node 'd8cc7b'. Skipping!
Property 'summary_embedding' already exists in node '26e922'. Skipping!
Property 'summary_embedding' already exists in node 'bc887b'. Skipping!
Property 'summary_embedding' already exists in node 'fd9a6f'. Skipping!
Property 'summary_embedding' already exists in node '06bd36'. Skipping!
Property 'summary_embedding' already exists in node '317765'. Skipping!
Property 'summary_embedding' already exists in node '03af45'. Skipping!
Property 'summary_embedding' already exists in node '0317bf'. Skipping!
Property 'summary_embedding' already exists in node '1490f9'. Skipping!
Property 'summary_embedding' already exists in node '4f9272'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Whaet is the knoledge centeR?,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not provide specific informat...,single_hop_specifc_query_synthesizer
1,Wha Chapter 3 is about and how does it relate ...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
2,What is the significance of Title IV in relati...,[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
3,What is Volume 8 in relation to disbursement t...,[both the credit or clock hours and the weeks ...,Volume 8 discusses how accelerated progression...,single_hop_specifc_query_synthesizer
4,How do disbursement timing requirements differ...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour or non-term credit-hour programs...,multi_hop_abstract_query_synthesizer
5,Considering the disbursement timing requiremen...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement timing rules differ significa...,multi_hop_abstract_query_synthesizer
6,How does the control over clinical work schedu...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work outside standar...,multi_hop_abstract_query_synthesizer
7,How do differences between standard and nonsta...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
8,How does Volume 8 explain the impact of accele...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 details that accelerated progression ...,multi_hop_specific_query_synthesizer
9,Based on the guidance provided in Volume 2 reg...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,The definitions of academic year and instructi...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [19]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [20]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [21]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [24]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available are Direct Subsidized Loans, Direct Unsubsidized Loans, Direct PLUS Loans (student Federal PLUS Loans), Subsidized and Unsubsidized Federal Stafford Loans, and Federal SLS Loans. However, Subsidized and Unsubsidized Federal Stafford Loans, Federal SLS Loans, and Federal PLUS Loans were made under the Federal Family Education Loan (FFEL) Program, which ended new loan authority effective July 1, 2010. New loans are made under the Direct Loan Program, including Direct Subsidized Loans, Direct Unsubsidized Loans, and Direct PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'crushing-floor-17' at:
https://smith.langchain.com/o/7a677cea-f3ba-437f-aa62-31066ac7d080/datasets/19aed3fe-aefb-4b0d-9065-97c90fc257ac/compare?selectedSessions=7c10ea28-b449-4c81-80dd-05fb11ad7cd5




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,"In Volume 2 and Volume 7, how does the academi...",Based on the provided context from Volume 2 an...,None,"According to Volume 2, the disbursement of fed...",1,1,0,8.002215,93865f3c-5248-4964-b982-dbedaa9e6b16,fe02a32c-d66d-456f-97b0-df62c17b4653
1,How do the disbursement timing rules differ be...,I don't know.,None,Volume 7 explains that Pell Grant and TEACH Gr...,0,0,0,0.606333,d881bb2a-fef0-45dd-9083-be32d6cfabe6,8c1ea5f0-6cfd-4440-916d-8dd74e7b6bff
2,Based on the guidance provided in Volume 2 reg...,Based on the provided context:\n\nThe definiti...,None,The definitions of academic year and instructi...,1,1,0,7.403217,f3fe504b-bddf-40fe-8c99-ecdbfd1055e4,564cc09d-6bef-412f-8f1f-3f233227af22
3,How does Volume 8 explain the impact of accele...,"Based on the provided context, Volume 8 discus...",None,Volume 8 details that accelerated progression ...,0,0,0,3.004916,0d9a8334-2545-4cb5-99ef-ce26550160a5,b7e172d0-0408-4e7c-80b4-648573c66f6e
4,How do differences between standard and nonsta...,"Based on the provided context, the differences...",None,The inclusion of clinical work in standard ter...,1,1,0,8.555381,ab04c6ca-4f55-4d54-acb9-b34f41efe920,9ce46147-0165-4a81-8c61-61799350f739
5,How does the control over clinical work schedu...,"Based on the context, clinical work that occur...",None,The inclusion of clinical work outside standar...,1,1,0,3.482188,70d42a2e-054d-40ae-b449-c76a1fa55b22,69a6bb75-d7de-4c71-99d3-b56f35ab577a
6,Considering the disbursement timing requiremen...,"Based on the provided context, the disbursemen...",None,The disbursement timing rules differ significa...,1,1,1,8.716047,79b32b92-ef03-427c-ae2a-0882f7b6e8c7,578ea939-829d-4f6c-a065-3b8c9ce50ada
7,How do disbursement timing requirements differ...,Based on the provided context:\n\n**Subscripti...,None,In clock-hour or non-term credit-hour programs...,1,1,0,5.724724,e62414f4-7f8b-4ea7-8716-8f389381b716,bfe6de3e-216d-4a18-b547-e173bcf92017
8,What is Volume 8 in relation to disbursement t...,"Based on the provided context, Volume 8 of the...",None,Volume 8 discusses how accelerated progression...,0,0,0,3.173817,cf8d7381-2171-4cd0-9f0f-d0439fa5271e,23c7c31b-7ed8-4b6c-83f4-db639041ed19
9,What is the significance of Title IV in relati...,Title IV programs require that disbursements (...,None,The payment period is applicable to all Title ...,1,1,0,2.418517,2d0536c0-0134-48b2-b50d-09fff58a3b0e,6bcdf0ab-5c06-44e7-af83-628cd652a70d


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Larger chunks contain more context per chunk, so this way we can loose less context. In our case we work with quite large dataset and complex queries, so contextual understanding is important and works better when provided with more comprehensive information.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

It has more dimensions, which means better semantic understanding and more nuanced matching.

In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question! Based on the information you've shared, there are several types of loans available to help with educational costs:\n\n1. **Direct Subsidized Loans** – These loans are based on financial need, and the government pays the interest while you’re in school. You can borrow up to the amount of your financial need (Cost of Attendance minus aid received).\n\n2. **Direct Unsubsidized Loans** – Available to students regardless of financial need, these loans accrue interest from the time they are disbursed. Students can borrow these loans to cover costs beyond the subsidized loan limits.\n\n3. **Direct PLUS Loans** – These loans are for parents of dependent students or for independent students themselves. Parents can take out a Direct PLUS Loan to cover the student's cost of attendance if they meet eligibility requirements. There is no fixed limit, but the loan cannot exceed the cost of attendance minus other financial aid.\n\nIt's important to know that if a parent i

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'shiny-son-44' at:
https://smith.langchain.com/o/7a677cea-f3ba-437f-aa62-31066ac7d080/datasets/19aed3fe-aefb-4b0d-9065-97c90fc257ac/compare?selectedSessions=e8bb24ad-012a-4864-b932-76b83117390b




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,"In Volume 2 and Volume 7, how does the academi...",Thank you for your thoughtful question. Based ...,None,"According to Volume 2, the disbursement of fed...",1,1,1,5.398058,93865f3c-5248-4964-b982-dbedaa9e6b16,8da90a2e-bfe8-4b86-973b-e9016e38db95
1,How do the disbursement timing rules differ be...,Thank you for your thoughtful question—it's cl...,None,Volume 7 explains that Pell Grant and TEACH Gr...,0,0,1,4.791009,d881bb2a-fef0-45dd-9083-be32d6cfabe6,bbb59c2c-5b3e-400c-bd41-81a9021a28f5
2,Based on the guidance provided in Volume 2 reg...,Thank you for your thoughtful question. Based ...,None,The definitions of academic year and instructi...,1,1,1,7.654061,f3fe504b-bddf-40fe-8c99-ecdbfd1055e4,fca30a09-6ca6-4baa-bf32-0a3223a47cc7
3,How does Volume 8 explain the impact of accele...,Thank you for your thoughtful question. From t...,None,Volume 8 details that accelerated progression ...,1,0,1,4.640382,0d9a8334-2545-4cb5-99ef-ce26550160a5,d0390ddc-d717-47a4-ab0e-0e54d0d7e287
4,How do differences between standard and nonsta...,Thank you for your thoughtful question. I can ...,None,The inclusion of clinical work in standard ter...,1,1,1,8.959308,ab04c6ca-4f55-4d54-acb9-b34f41efe920,e27d8107-4f37-41bb-9c37-1636dbefb5aa
5,How does the control over clinical work schedu...,Thank you for your thoughtful question. Based ...,None,The inclusion of clinical work outside standar...,1,1,1,4.917691,70d42a2e-054d-40ae-b449-c76a1fa55b22,388683dc-5ed8-42b5-a987-0ec31ff539b0
6,Considering the disbursement timing requiremen...,Thank you for your thoughtful question. I can ...,None,The disbursement timing rules differ significa...,1,1,1,7.991198,79b32b92-ef03-427c-ae2a-0882f7b6e8c7,c1b3c9cc-af17-4e95-935e-71911218917f
7,How do disbursement timing requirements differ...,Thank you for your thoughtful question—it's cl...,None,In clock-hour or non-term credit-hour programs...,1,0,1,6.964546,e62414f4-7f8b-4ea7-8716-8f389381b716,c25d1854-ba19-49ae-a5c7-b1ed1bc1c387
8,What is Volume 8 in relation to disbursement t...,Thank you for your thoughtful question. Based ...,None,Volume 8 discusses how accelerated progression...,0,0,1,4.160615,cf8d7381-2171-4cd0-9f0f-d0439fa5271e,25258667-a150-4c90-9baf-3663007f394c
9,What is the significance of Title IV in relati...,I understand you're looking to grasp the impor...,None,The payment period is applicable to all Title ...,1,1,1,7.949393,2d0536c0-0134-48b2-b50d-09fff58a3b0e,8a3af35e-a22c-48ac-a2d3-bdaf0ca2897c


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

| run 1 | run 2 |
|---------|--------|
| ![before](./eval-before.png) | ![after](./eval-after.png) |

We can see that empathy score much higher (moving from 0 to 1) because the prompt explicitly instructs empathetic responses.

Correctness went a bit higher because larger chunks and better embeddings provided more complete context, but in general they maintained pretty similar accuracy levels.

I am not sure why helpfulness wasn't affected, probably because the casual tone potentially affecting clarity (because of empathy optimization?)

